In [2]:
!pip install -q "transformers>=4.40.0" "datasets>=2.19.0" "accelerate" "evaluate"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


In [3]:
import torch
import numpy as np

from datasets import load_dataset
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

In [4]:
# -------------------------------------------------------------------
# 1. Load full GoEmotions (multi-label, 28 emotions + neutral)
# -------------------------------------------------------------------
dataset = load_dataset("go_emotions")   # train / validation / test
print(dataset)
print("Example row:", dataset["train"][0])

fine_label_names = dataset["train"].features["labels"].feature.names
print("Fine labels:", fine_label_names)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

simplified/train-00000-of-00001.parquet:   0%|          | 0.00/2.77M [00:00<?, ?B/s]

simplified/validation-00000-of-00001.par(…):   0%|          | 0.00/350k [00:00<?, ?B/s]

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})
Example row: {'text': "My favourite food is anything I didn't have to cook myself.", 'labels': [27], 'id': 'eebbqej'}
Fine labels: ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


In [5]:
# -------------------------------------------------------------------
# 2. Define 6 main emotions and mapping 28 → 6
# -------------------------------------------------------------------
six_labels = ["joy", "anger", "sadness", "fear", "surprise", "neutral"]
six_label2id = {name: i for i, name in enumerate(six_labels)}
id2six_label = {i: name for name, i in six_label2id.items()}
print("Six labels:", six_label2id)

# mapping from fine label name → 6-way label name
to_six = {
    "admiration": "joy",
    "amusement": "joy",
    "approval": "joy",
    "excitement": "joy",
    "gratitude": "joy",
    "joy": "joy",
    "love": "joy",
    "optimism": "joy",
    "pride": "joy",
    "relief": "joy",

    "anger": "anger",
    "annoyance": "anger",
    "disapproval": "anger",

    "disappointment": "sadness",
    "disgust": "sadness",
    "embarrassment": "sadness",
    "grief": "sadness",
    "remorse": "sadness",
    "sadness": "sadness",

    "confusion": "fear",
    "fear": "fear",
    "nervousness": "fear",

    "realization": "surprise",
    "surprise": "surprise",
    "curiosity": "surprise",

    "caring": "neutral",
    "desire": "neutral",
    "neutral": "neutral",
}

# id -> fine label name
id2fine = {i: name for i, name in enumerate(fine_label_names)}

def map_to_six(example):
    """
    GoEmotions is multi-label (list of fine label IDs).
    For simplicity we:
      - if no labels: neutral
      - else: take the FIRST fine label and map it to one of 6 classes.
    """
    labs = example["labels"]  # list of fine-label IDs
    if len(labs) == 0:
        fine = "neutral"
    else:
        fine = id2fine[labs[0]]
    big = to_six[fine]
    example["label6"] = six_label2id[big]
    return example

dataset6 = dataset.map(map_to_six)
print("After mapping, example:", dataset6["train"][0])

# keep only text + 6-class label
cols_to_keep = ["text", "label6"]
dataset6 = dataset6.remove_columns(
    [c for c in dataset6["train"].column_names if c not in cols_to_keep]
)
dataset6 = dataset6.rename_column("label6", "labels")

print("Columns after cleanup:", dataset6["train"].column_names)
print("Example row:", dataset6["train"][0])

Six labels: {'joy': 0, 'anger': 1, 'sadness': 2, 'fear': 3, 'surprise': 4, 'neutral': 5}


Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

After mapping, example: {'text': "My favourite food is anything I didn't have to cook myself.", 'labels': [27], 'id': 'eebbqej', 'label6': 5}
Columns after cleanup: ['text', 'labels']
Example row: {'text': "My favourite food is anything I didn't have to cook myself.", 'labels': 5}


In [6]:
# ---- Export GoEmotions test split to JSONL for local use ----

six_labels = ["joy", "anger", "sadness", "fear", "surprise", "neutral"]
id2six_label = {i: lab for i, lab in enumerate(six_labels)}

def add_label_name(example):
    example["gold_label"] = id2six_label[example["labels"]]
    return example

# add a human-readable gold label
dataset6_named = dataset6.map(add_label_name)

# Keep only text & gold_label
dataset6_named = dataset6_named.remove_columns(
    [c for c in dataset6_named["train"].column_names if c not in ["text", "gold_label"]]
)

# Save the test split as JSONL (one JSON object per line)
dataset6_named["test"].to_json("goemotions_test.jsonl", lines=True)

from google.colab import files
files.download("goemotions_test.jsonl")


Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/6 [00:00<?, ?ba/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# -------------------------------------------------------------------
# 3. Tokenizer & model
# -------------------------------------------------------------------
model_name = "bert-base-uncased"  

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(six_labels),
    id2label=id2six_label,
    label2id=six_label2id,
)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
# -------------------------------------------------------------------
# 4. Tokenization / preprocessing
# -------------------------------------------------------------------
def preprocess(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=128,
    )

encoded_dataset6 = dataset6.map(
    preprocess,
    batched=True,
)
print("Encoded example:", encoded_dataset6["train"][0])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

Encoded example: {'text': "My favourite food is anything I didn't have to cook myself.", 'labels': 5, 'input_ids': [0, 2387, 5548, 689, 16, 932, 38, 399, 75, 33, 7, 7142, 2185, 4, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [7]:
# -------------------------------------------------------------------
# 5. Metrics
# -------------------------------------------------------------------
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }


In [ ]:
# -------------------------------------------------------------------
# 6. Train/val/test split
# -------------------------------------------------------------------
train_dataset = encoded_dataset6["train"]
val_dataset   = encoded_dataset6["validation"]
test_dataset  = encoded_dataset6["test"]

batch_size = 16

training_args = TrainingArguments(
    output_dir="./berta-goemotions-6class",
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=1000,
    # older versions don't support evaluation_strategy/save_strategy/etc.
    fp16=torch.cuda.is_available(),   # mixed precision if GPU available
    report_to="none",                 # no wandb
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,   # still used during training for periodic eval if supported
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


/tmp/ipython-input-416612949.py:24: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [10]:
# -------------------------------------------------------------------
# 7. Training
# -------------------------------------------------------------------
trainer.train()

print("Validation results (best model):")
val_results = trainer.evaluate(val_dataset)
print(val_results)



print("\nTest results:")
test_results = trainer.evaluate(test_dataset)
print(test_results)


Step,Training Loss
1000,1.058800
2000,0.908000
3000,0.852300
4000,0.784300
5000,0.760400
6000,0.719300
7000,0.654800
8000,0.645400


Validation results (best model):


{'eval_loss': 0.8745025396347046, 'eval_accuracy': 0.6887209730925176, 'eval_f1_macro': 0.617405537856145, 'eval_runtime': 5.5764, 'eval_samples_per_second': 973.033, 'eval_steps_per_second': 60.971, 'epoch': 3.0}

Test results:
{'eval_loss': 0.8869246244430542, 'eval_accuracy': 0.6838032061912659, 'eval_f1_macro': 0.6119743245388597, 'eval_runtime': 5.4692, 'eval_samples_per_second': 992.283, 'eval_steps_per_second': 62.166, 'epoch': 3.0}


In [ ]:
# -------------------------------------------------------------------
# 8. Save final model + tokenizer
# -------------------------------------------------------------------
save_dir = "./berta-goemotions-6class-final"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

print("Saved to:", save_dir)


Saved to: ./roberta-goemotions-6class-final


In [12]:
# -------------------------------------------------------------------
# 9. Simple prediction function for unseen text
# -------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

id2label_6 = id2six_label  # just for clarity

def predict_emotion_6(text: str):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)[0]
        pred_id = int(torch.argmax(probs).item())

    return {
        "text": text,
        "emotion": id2label_6[pred_id],
        "score": float(probs[pred_id].item()),
    }

print(predict_emotion_6("The traffic in Boston is driving me crazy today."))
print(predict_emotion_6("I am so happy about the game last night!"))

{'text': 'The traffic in Boston is driving me crazy today.', 'emotion': 'joy', 'score': 0.7148774862289429}
{'text': 'I am so happy about the game last night!', 'emotion': 'joy', 'score': 0.9952360987663269}


In [13]:
# zip the folder
!zip -r roberta-goemotions-6class-final.zip roberta-goemotions-6class-final

from google.colab import files
files.download("roberta-goemotions-6class-final.zip")


  adding: roberta-goemotions-6class-final/ (stored 0%)
  adding: roberta-goemotions-6class-final/tokenizer_config.json (deflated 75%)
  adding: roberta-goemotions-6class-final/training_args.bin (deflated 53%)
  adding: roberta-goemotions-6class-final/config.json (deflated 53%)
  adding: roberta-goemotions-6class-final/vocab.json (deflated 59%)
  adding: roberta-goemotions-6class-final/model.safetensors (deflated 10%)
  adding: roberta-goemotions-6class-final/tokenizer.json (deflated 82%)
  adding: roberta-goemotions-6class-final/merges.txt (deflated 53%)
  adding: roberta-goemotions-6class-final/special_tokens_map.json (deflated 52%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
!pip install wordcloud matplotlib pandas


In [ ]:
import json
import pandas as pd
from tqdm import tqdm

from test_model import predict_emotion_6

INPUT_PATH = "goemotions_test.jsonl"
N = 800   # number of comments to visualize

rows = []

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    for i, line in enumerate(tqdm(f, total=N)):
        if i >= N:
            break
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        text = obj["text"]
        gold = obj.get("gold_label")

        pred = predict_emotion_6(text)
        rows.append({
            "text": text,
            "gold": gold,
            "pred": pred["emotion"],
            "score": pred["score"],
        })

df = pd.DataFrame(rows)
df.head()


In [ ]:
import matplotlib.pyplot as plt

counts = df["pred"].value_counts()

plt.figure(figsize=(6,4))
counts.plot(kind="bar")
plt.title("Predicted Emotion Distribution")
plt.xlabel("Emotion")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
from wordcloud import WordCloud

def show_wordcloud_for_emotion(emotion_label):
    subset = df[df["pred"] == emotion_label]
    text_blob = " ".join(subset["text"].astype(str).tolist())

    wc = WordCloud(
        width=1000,
        height=500,
        background_color="white"
    ).generate(text_blob)

    plt.figure(figsize=(10,5))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"Word Cloud for '{emotion_label}'")
    plt.show()

show_wordcloud_for_emotion("joy")


In [ ]:
for emo in ["joy", "anger", "sadness", "fear", "surprise", "neutral"]:
    show_wordcloud_for_emotion(emo)
